# 🛰️ NN_segmentation — Colab Launcher

Questo notebook gestisce l'intero pipeline:
1. **Clona / aggiorna** il codice da GitHub
2. **Installa** le dipendenze e i pesi pre-addestrati
3. **Monta Google Drive** e recupera il dataset OpenEarthMap
4. **Prepara il subset intelligente** (solo la prima volta, ~15 min)
5. **Avvia il training**

---
### 📂 Struttura attesa su Google Drive
```
MyDrive/
└── NN_Datasets/
    └── OpenEarthMap/
        ├── open_earth_map.zip        ← dataset raw (caricato da te una volta)
        ├── oem_train_indices.json    ← generato automaticamente (1a run)
        └── oem_val_indices.json      ← generato automaticamente (1a run)
```
I file JSON vengono creati e **salvati su Drive** alla prima esecuzione;
dalla seconda in poi vengono solo copiati in locale (istantaneo).

## ① Clone / Pull del codice da GitHub

In [ ]:
import os
REPO_URL  = 'https://github.com/robertopassante/NN_segmentation.git'
REPO_NAME = 'NN_segmentation'

if not os.path.exists(f'/content/{REPO_NAME}'):
    print(f'[INFO] Clonazione del repository {REPO_NAME} in corso...')
    !git clone {REPO_URL}
    %cd {REPO_NAME}
else:
    print(f'[INFO] Repository già presente. Aggiornamento in corso...')
    %cd /content/{REPO_NAME}
    !git pull

print('[OK] Codice pronto.')

## ② Installazione dipendenze e pesi pre-addestrati

In [ ]:
!pip install -q -r requirements.txt
!pip install -q yacs safetensors

# Scarica i pesi Swin-T pre-addestrati su immagini satellitari (RSP)
if not os.path.exists('rsp-swin-t-ckpt.safetensors'):
    print('[INFO] Download pesi RSP Swin-T...')
    !wget -q -O rsp-swin-t-ckpt.safetensors \
        https://huggingface.co/BiliSakura/RSP-Swin-T/resolve/main/model.safetensors
    print('[OK] Pesi scaricati.')
else:
    print('[SKIP] Pesi RSP già presenti.')

## ③ Mount Google Drive + ripristino dataset

> **Cosa succede qui:**
> - Il dataset **non viene mai ri-scaricato** da internet: è già nel tuo Drive.
> - Lo zip viene copiato da Drive → `/content/dataset/` ed estratto (solo se non è già estratto).
> - I file JSON degli indici vengono copiati da Drive (se esistono già) oppure generati dopo.

In [ ]:
import os, shutil
from google.colab import drive

# ── Monta Drive ─────────────────────────────────────────────────────────────
drive.mount('/content/drive')

DRIVE_OEM = '/content/drive/MyDrive/NN_Datasets/OpenEarthMap'
DATA_DIR  = '/content/NN_segmentation/dataset'
os.makedirs(DATA_DIR, exist_ok=True)

# ── Estrai dataset raw (solo se non già fatto) ───────────────────────────────
ZIP_NAME    = 'open_earth_map.zip'
zip_src     = os.path.join(DRIVE_OEM, ZIP_NAME)
zip_dst     = os.path.join(DATA_DIR,  ZIP_NAME)
oem_folder  = os.path.join(DATA_DIR,  'OpenEarthMap')

if not os.path.exists(oem_folder):
    if not os.path.exists(zip_dst):
        print(f'[INFO] Copio {ZIP_NAME} da Google Drive...')
        shutil.copy(zip_src, zip_dst)
        print(f'[OK]  {ZIP_NAME} copiato ({os.path.getsize(zip_dst)/1e9:.1f} GB)')
    print('[INFO] Estrazione in corso (può richiedere qualche minuto)...')
    %cd {DATA_DIR}
    !unzip -q {ZIP_NAME}
    print('[OK]  Estrazione completata.')
else:
    print('[SKIP] Dataset OpenEarthMap già estratto in', oem_folder)

# ── Ripristina i JSON degli indici da Drive (se già generati in passato) ─────
for json_name in ['oem_train_indices.json', 'oem_val_indices.json']:
    src = os.path.join(DRIVE_OEM, json_name)
    dst = os.path.join(DATA_DIR,  json_name)
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy(src, dst)
        print(f'[OK]  Indici {json_name} ripristinati da Drive.')
    elif os.path.exists(dst):
        print(f'[SKIP] {json_name} già presente in locale.')
    else:
        print(f'[INFO] {json_name} non trovato su Drive (verrà generato al passo ④).')

%cd /content/NN_segmentation
print('\n[SUCCESS] Dataset OpenEarthMap pronto!')

## ④ Preparazione Smart Subset (solo alla prima esecuzione)

> **Cosa fa:**  
> Analizza ogni immagine del dataset e seleziona quelle con una classe dominante ≥ 40%.  
> Mantiene al massimo **150 immagini/classe** per il train e **40/classe** per la val.  
> Il risultato viene salvato in due file JSON su Drive così non si ripete mai più.
>
> **Durata stimata:** ~15-20 minuti (solo la prima volta).  
> **Dalla 2ª volta in poi:** i JSON sono già copiati da Drive → questa cella termina in <5 secondi.

In [ ]:
import os, shutil

DRIVE_OEM = '/content/drive/MyDrive/NN_Datasets/OpenEarthMap'
DATA_DIR  = '/content/NN_segmentation/dataset'

train_json_local = os.path.join(DATA_DIR,  'oem_train_indices.json')
val_json_local   = os.path.join(DATA_DIR,  'oem_val_indices.json')
train_json_drive = os.path.join(DRIVE_OEM, 'oem_train_indices.json')
val_json_drive   = os.path.join(DRIVE_OEM, 'oem_val_indices.json')

both_exist = os.path.exists(train_json_local) and os.path.exists(val_json_local)

if both_exist:
    print('✅ Indici già presenti — subset pronto, skip preparazione.')
else:
    print('🔍 Generazione smart subset in corso...')
    print('   (analisi di ogni mask — ~15-20 min, solo questa volta)')
    print('─' * 60)
    !python data/prepare_dataset.py
    print('─' * 60)

    # ── Salva i JSON su Drive per le esecuzioni future ────────────────────
    os.makedirs(DRIVE_OEM, exist_ok=True)
    for local, drive_path in [(train_json_local, train_json_drive),
                               (val_json_local,   val_json_drive)]:
        if os.path.exists(local):
            shutil.copy(local, drive_path)
            print(f'[OK]  Salvato su Drive: {os.path.basename(drive_path)}')
        else:
            print(f'[WARN] File non trovato: {local}')

    print('\n✅ Subset intelligente pronto e salvato su Drive!')

## ⑤ Training

In [ ]:
%cd /content/NN_segmentation
!python main.py